# 03 Conversational Explanations

This notebook turns local GNNExplainer outputs into grounded, human-readable conversational explanations.

## 1. Setup

Import the required libraries, configure the notebook, and define the file paths used for loading GNNExplainer outputs and saving conversational explanations.

#### 1.1 Import Libraries

In [1]:
import json
import os
from pathlib import Path

import pandas as pd
import rdflib as rdf
from dotenv import load_dotenv
from google import genai
from google.genai import types
from google.genai.errors import ClientError
from rdflib import URIRef

from graph_llm_explainer import (
    SWRC,
    GraphDecoder,
    build_grounded_prompt,
    build_model_triples,
    build_relation_vocabulary,
)

#### 1.2 Configure Notebook

In [2]:
USE_LLM = True
GEMINI_MODEL = "gemini-2.5-flash"
TOP_K_EVIDENCE = 10

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if USE_LLM and not GEMINI_API_KEY:
    raise RuntimeError(
        "Set the GEMINI_API_KEY environment variable before running this notebook."
    )

#### 1.3 Define File Paths

In [3]:
# Locate the project root relative to the working directory
PROJECT_ROOT = Path(".")

if not (PROJECT_ROOT / "data" / "raw").exists():
    PROJECT_ROOT = Path("..")

# Project directories
DATA_DIR = PROJECT_ROOT / "data" / "raw"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
RESULTS_DIR = PROJECT_ROOT / "results"
EXPLANATIONS_DIR = RESULTS_DIR / "explanations"
CONVERSATIONS_DIR = RESULTS_DIR / "conversations"

# Dataset files
RDF_PATH = DATA_DIR / "aifbfixed_complete.n3"
COMPLETE_LABELS_PATH = DATA_DIR / "completeDataset.tsv"
TRAIN_LABELS_PATH = DATA_DIR / "trainingSet.tsv"
TEST_LABELS_PATH = DATA_DIR / "testSet.tsv"

# Model checkpoint
MODEL_CHECKPOINT_PATH = ARTIFACT_DIR / "rgcn_baseline.pt"

# Create output directories if needed
EXPLANATIONS_DIR.mkdir(parents=True, exist_ok=True)
CONVERSATIONS_DIR.mkdir(parents=True, exist_ok=True)

# Verify required files and directories
required_paths = {
    "RDF graph": RDF_PATH,
    "Complete labels": COMPLETE_LABELS_PATH,
    "Training split": TRAIN_LABELS_PATH,
    "Test split": TEST_LABELS_PATH,
    "Model checkpoint": MODEL_CHECKPOINT_PATH,
    "Explanations": EXPLANATIONS_DIR,
    "Conversations": CONVERSATIONS_DIR,
}

for name, path in required_paths.items():
    print(f"{name:<18} {path.exists()}  {path}")

RDF graph          True  ../data/raw/aifbfixed_complete.n3
Complete labels    True  ../data/raw/completeDataset.tsv
Training split     True  ../data/raw/trainingSet.tsv
Test split         True  ../data/raw/testSet.tsv
Model checkpoint   True  ../artifacts/rgcn_baseline.pt
Explanations       True  ../results/explanations
Conversations      True  ../results/conversations


## 2. Reconstruct RDF Graph Encoding

The saved GNNExplainer files contain numeric node and edge identifiers. Therefore, the RDF graph encoding from Notebook 02 is reconstructed so that explanation edges can be decoded back into RDF-style facts.


#### 2.1 Load RDF Graph

In [4]:
# Load the complete RDF knowledge graph
g = rdf.Graph().parse(RDF_PATH, format="n3")


#### 2.2 Recreate Model Triples

The affiliation target and the `member` and `employs` shortcut relations are removed, exactly as in Notebook 02. The relation vocabulary below must match the trained model, otherwise edge types decode to the wrong predicates.

In [5]:
# Remove the predicates that were excluded when the R-GCN was trained
model_triples = build_model_triples(g)


#### 2.3 Define Readable Name Predicates

In [6]:
# Predicates used to resolve RDF resources into readable names
NAME_PREDICATES = [
    URIRef(SWRC + "name"),
    URIRef(SWRC + "title"),
    URIRef("http://www.w3.org/2000/01/rdf-schema#label"),
]

#### 2.4 Rebuild Node and Relation Mappings

In [7]:
# Recreate stable relation and node mappings from the model graph
relations = build_relation_vocabulary(model_triples)

nodes = sorted(
    set(g.subjects()).union(g.objects()),
    key=str,
)

relations_dict = {relation: idx for idx, relation in enumerate(relations)}

nodes_dict = {str(node): idx for idx, node in enumerate(nodes)}

inv_nodes_dict = {idx: str(node) for node, idx in nodes_dict.items()}

inv_relations_dict = {idx: relation for relation, idx in relations_dict.items()}

#### 2.5 Rebuild Encoded Edge Table

In [8]:
# Recreate forward and inverse edges used by the R-GCN
edge_rows = []

for subject, predicate, obj in model_triples:
    source_id = nodes_dict[str(subject)]
    target_id = nodes_dict[str(obj)]
    relation_id = relations_dict[predicate]

    edge_rows.extend(
        [
            {
                "source_id": source_id,
                "target_id": target_id,
                "edge_type": 2 * relation_id,
                "direction": "forward",
            },
            {
                "source_id": target_id,
                "target_id": source_id,
                "edge_type": 2 * relation_id + 1,
                "direction": "inverse",
            },
        ]
    )

edge_table = pd.DataFrame(edge_rows)

#### 2.6 Load Label Mappings

In [9]:
# Load labels and recreate class mappings
label_header = "label_affiliation"
node_header = "person"

labels_df = pd.read_csv(COMPLETE_LABELS_PATH, sep="\t")
train_labels_df = pd.read_csv(TRAIN_LABELS_PATH, sep="\t")
test_labels_df = pd.read_csv(TEST_LABELS_PATH, sep="\t")

labels = sorted(labels_df[label_header].unique())

labels_dict = {label: idx for idx, label in enumerate(labels)}

inv_labels_dict = {idx: label for label, idx in labels_dict.items()}

#### 2.7 Validate Reconstructed Encoding

In [10]:
encoding_reconstruction_summary = pd.Series(
    {
        "RDF triples": len(g),
        "Model triples": len(model_triples),
        "Nodes": len(nodes),
        "RDF relations used": len(relations),
        "Edge types with inverse": 2 * len(relations),
        "Labelled people": len(labels_df),
        "Classes": len(labels),
    },
    name="value",
).to_frame()

encoding_reconstruction_summary

,value
RDF triples,29226
Model triples,28704
Nodes,8285
RDF relations used,44
Edge types with inverse,88
Labelled people,176
Classes,4


## 3. Prepare Explanation Utilities

This section creates the decoder that turns saved GNNExplainer outputs into readable RDF facts. The grounded prompt is built by `build_grounded_prompt` from the package.


#### 3.1 Create the Fact Decoder

The `graph_llm_explainer` package decodes numeric node and edge-type identifiers into RDF facts (`facts.py`) and builds the grounded prompt (`prompts.py`).

In [11]:
decoder = GraphDecoder(
    g,
    inv_nodes_dict,
    relations,
    name_predicates=NAME_PREDICATES,
)


## 4. Load Saved GNN Explanations

This section loads the GNNExplainer artifacts generated in the previous notebook and converts them into a common record format.

#### 4.1 Normalize Explanation Records

In [12]:
def normalize_explanation_record(record):
    """Convert one explanation record into the common explanation format."""
    record = dict(record)

    if "node_id" in record:
        record["node_id"] = int(record["node_id"])

    # Convert single-edge records into an edge list
    if "edges" not in record and {"source_id", "target_id", "edge_type"}.issubset(
        record
    ):
        record["edges"] = [
            {
                "source_id": int(record["source_id"]),
                "target_id": int(record["target_id"]),
                "edge_type": int(record["edge_type"]),
                "importance": float(record.get("importance", record.get("score", 0.0))),
            }
        ]

    return record

#### 4.2 Convert Tabular Explanation Files

In [13]:
def records_from_dataframe(df):
    """Convert tabular explanation data into normalized explanation records."""

    if "node_id" in df.columns and {"source_id", "target_id", "edge_type"}.issubset(
        df.columns
    ):
        records = []

        metadata_columns = [
            column
            for column in df.columns
            if column
            not in {
                "source_id",
                "target_id",
                "edge_type",
                "importance",
                "score",
            }
        ]

        for node_id, group in df.groupby("node_id"):
            first = group.iloc[0]

            record = {
                column: first[column] for column in metadata_columns if column in first
            }

            record["node_id"] = int(node_id)
            record["edges"] = [
                {
                    "source_id": int(row["source_id"]),
                    "target_id": int(row["target_id"]),
                    "edge_type": int(row["edge_type"]),
                    "importance": float(row.get("importance", row.get("score", 0.0))),
                }
                for _, row in group.iterrows()
            ]

            records.append(record)

        return records

    return [normalize_explanation_record(row) for row in df.to_dict(orient="records")]

#### 4.3 Load Explanation Files

In [14]:
def load_saved_explanations(explanations_dir):
    """Load and normalize all supported explanation files from a directory."""
    explanations = []

    files = sorted(
        list(explanations_dir.glob("*.json"))
        + list(explanations_dir.glob("*.jsonl"))
        + list(explanations_dir.glob("*.csv"))
        + list(explanations_dir.glob("*.parquet"))
    )

    for path in files:
        if path.name.startswith("conversation_"):
            continue

        if path.suffix == ".json":
            payload = json.loads(path.read_text())

            if isinstance(payload, dict) and "explanations" in payload:
                items = payload["explanations"]
            elif isinstance(payload, list):
                items = payload
            else:
                items = [payload]

            explanations.extend(normalize_explanation_record(item) for item in items)

        elif path.suffix == ".jsonl":
            for line in path.read_text().splitlines():
                if line.strip():
                    explanations.append(normalize_explanation_record(json.loads(line)))

        elif path.suffix == ".csv":
            explanations.extend(records_from_dataframe(pd.read_csv(path)))

        elif path.suffix == ".parquet":
            explanations.extend(records_from_dataframe(pd.read_parquet(path)))

    return explanations

#### 4.4 Remove Duplicate Explanation Records

In [15]:
def deduplicate_explanations(explanations):
    """Keep one explanation record per explained node."""
    deduplicated = []
    seen_node_ids = set()

    for explanation in explanations:
        node_id = explanation.get("node_id")

        if node_id is not None:
            node_id = int(node_id)

        if node_id in seen_node_ids:
            continue

        seen_node_ids.add(node_id)
        deduplicated.append(explanation)

    return deduplicated

#### 4.5 Load Available Explanations

In [16]:
explanations = load_saved_explanations(EXPLANATIONS_DIR)
explanations = deduplicate_explanations(explanations)

print(f"Loaded {len(explanations)} saved explanation record(s).")

if not explanations:
    raise FileNotFoundError(
        f"No saved explanation records found in {EXPLANATIONS_DIR}.\n"
        "Please run Notebook 02 first to generate GNNExplainer artifacts."
    )

Loaded 34 saved explanation record(s).


## 5. Reconstruct Human-Readable Explanations

This section converts a saved GNNExplainer record into two human-readable objects: a ranked evidence table and a standardized prediction summary.

#### 5.1 Select Explanation Record

In [17]:
example_explanation = explanations[18]

#### 5.2 Build Evidence Table

In [18]:
evidence_table = decoder.build_evidence_table(
    example_explanation,
    top_k=TOP_K_EVIDENCE,
)

evidence_table[
    [
        "source_label",
        "target_label",
        "predicate",
        "direction",
        "importance_score",
        "generated_fact",
    ]
]

,source_label,target_label,predicate,direction,importance_score,generated_fact
0,+49 (721) 608 4548,Yu Li,:fax,inverse,0.739084,Yu Li --:fax--> +49 (721) 608 4548
1,Ein Framework zur Modellierung und Analyse von...,Yu Li,:publication,inverse,0.673273,Yu Li --:publication--> Ein Framework zur Mode...
2,Simulation von Supply Chains mit Petri-Netzen,Yu Li,:publication,inverse,0.656043,Yu Li --:publication--> Simulation von Supply ...
3,Business Information and Communication Systems,Ein Framework zur Modellierung und Analyse von...,:publishes,forward,0.470060,Business Information and Communication Systems...
4,Business Information and Communication Systems,Simulation von Supply Chains mit Petri-Netzen,:publishes,forward,0.450062,Business Information and Communication Systems...
5,Ein Framework zur Modellierung und Analyse von...,Yu Li,:author,forward,0.415170,Ein Framework zur Modellierung und Analyse von...
6,Prozessorientierte Komposition von Diensten in...,Yu Li,:publication,inverse,0.406716,Yu Li --:publication--> Prozessorientierte Kom...
7,Business Information and Communication Systems,Prozessorientierte Komposition von Diensten in...,:publishes,forward,0.377735,Business Information and Communication Systems...
8,Simulation von Supply Chains mit Petri-Netzen,Yu Li,:author,forward,0.358356,Simulation von Supply Chains mit Petri-Netzen ...
9,Yu Li,Yu Li,:name,inverse,0.337610,Yu Li --:name--> Yu Li


#### 5.3 Extract Prediction Metadata

In [19]:
def prediction_from_explanation(explanation):
    """Convert a saved explanation record into readable prediction metadata."""
    node_id = int(explanation["node_id"])
    node_uri = explanation.get("node_uri", inv_nodes_dict[node_id])

    predicted_label_uri = explanation.get("predicted_label_uri") or explanation.get(
        "pred_label_uri"
    )

    true_label_uri = explanation.get("true_label_uri")

    if predicted_label_uri is None and "predicted_label_id" in explanation:
        predicted_label_uri = inv_labels_dict[int(explanation["predicted_label_id"])]

    if true_label_uri is None and "true_label_id" in explanation:
        true_label_uri = inv_labels_dict[int(explanation["true_label_id"])]

    # Recover missing true label from the original label table if needed
    if predicted_label_uri is None or true_label_uri is None:
        match = labels_df[labels_df[node_header] == node_uri]

        true_label_uri = true_label_uri or (
            match.iloc[0][label_header] if len(match) else "unknown"
        )

        predicted_label_uri = predicted_label_uri or "unknown"

    return {
        "node_id": node_id,
        "node_uri": node_uri,
        "node_label": decoder.readable_node_label(node_id),
        "predicted_label_uri": predicted_label_uri,
        "true_label_uri": true_label_uri,
        "predicted_label": (
            decoder.readable_label(predicted_label_uri)
            if predicted_label_uri != "unknown"
            else "unknown"
        ),
        "true_label": (
            decoder.readable_label(true_label_uri)
            if true_label_uri != "unknown"
            else "unknown"
        ),
        "correct": bool(
            explanation.get(
                "correct",
                predicted_label_uri == true_label_uri,
            )
        ),
    }

#### 5.4 Inspect Prediction Summary

In [20]:
prediction = prediction_from_explanation(example_explanation)

prediction_summary = pd.Series(
    prediction,
    name="value",
).to_frame()

prediction_summary

,value
node_id,5672
node_uri,http://www.aifb.uni-karlsruhe.de/Personen/view...
node_label,Yu Li
predicted_label_uri,http://www.aifb.uni-karlsruhe.de/Forschungsgru...
true_label_uri,http://www.aifb.uni-karlsruhe.de/Forschungsgru...
predicted_label,Business Information and Communication Systems
true_label,Business Information and Communication Systems
correct,True


## 6. Inspect a Grounded Prompt

This section builds one example prompt to verify what information is passed to the LLM before generating conversational explanations.

In [21]:
example_question = "Why was this researcher assigned to this affiliation?"

example_prompt = build_grounded_prompt(
    prediction,
    evidence_table,
    example_question,
)

print(example_prompt)

You are a grounded conversational XAI assistant.

Task:
Answer the user's question about one node-classification prediction.

Prediction metadata:
- Explained node: Yu Li (http://www.aifb.uni-karlsruhe.de/Personen/viewPersonOWL/id2136instance)
- Predicted class: Business Information and Communication Systems
- True class: Business Information and Communication Systems
- Prediction correct: True

Grounding evidence from GNNExplainer:
1. Yu Li --:fax--> +49 (721) 608 4548 (direction: inverse; importance: 0.739)
2. Yu Li --:publication--> Ein Framework zur Modellierung und Analyse von XML-Netzen (direction: inverse; importance: 0.673)
3. Yu Li --:publication--> Simulation von Supply Chains mit Petri-Netzen (direction: inverse; importance: 0.656)
4. Business Information and Communication Systems --:publishes--> Ein Framework zur Modellierung und Analyse von XML-Netzen (direction: forward; importance: 0.470)
5. Business Information and Communication Systems --:publishes--> Simulation von Su

## 7. Generate a Simulated Conversation

This section simulates a realistic conversation about the selected prediction. Each answer is generated from the same prediction metadata and GNNExplainer evidence, which keeps the conversation grounded in the extracted graph facts.

> **Note on the recorded run.** The conversations stored in `results/conversations/` were generated before the relation vocabulary was aligned with Notebook 02. In that run 20 of the 60 evidence facts carry a wrong predicate label (for example `:homepage` instead of `:fax`). The code above decodes edges with the correct vocabulary; the answers and manual scores shown from here on are the recorded results and are not regenerated.


#### 7.1 Define Conversation Questions

In [22]:
conversation_questions = [
    "Why was this researcher assigned to this affiliation?",
    "Which graph relationship was most important for this prediction?",
    "Is this evidence strong enough to trust the prediction?",
    "Are there any weak or uninformative facts in the explanation?",
    "Can you summarize the explanation in one sentence?",
    "What can you not conclude from this evidence?",
]

#### 7.2 Define Gemini Helper

In [23]:
def ask_gemini(prompt, model=GEMINI_MODEL):
    """Send a grounded prompt to Gemini and return the generated answer."""
    if not USE_LLM:
        raise RuntimeError("Set USE_LLM = True to generate LLM responses.")

    client = genai.Client(api_key=GEMINI_API_KEY)

    try:
        response = client.models.generate_content(
            model=model,
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.2,
            ),
        )

    except ClientError as exc:
        if "RESOURCE_EXHAUSTED" in str(exc):
            raise RuntimeError(
                "Gemini API quota exceeded. Try again later or use another API key/project."
            ) from exc
        raise

    if not response.text:
        raise RuntimeError("Gemini returned no text response.")

    return response.text

#### 7.3 Generate Conversation

In [24]:
conversation_records = []

for turn_id, question in enumerate(conversation_questions, start=1):
    prompt = build_grounded_prompt(
        prediction,
        evidence_table,
        question,
    )

    try:
        answer = ask_gemini(prompt)
        error = None

    except Exception as exc:
        answer = None
        error = f"{type(exc).__name__}: {exc}"

    conversation_records.append(
        {
            "turn": turn_id,
            "node_id": prediction["node_id"],
            "node_label": prediction["node_label"],
            "predicted_label": prediction["predicted_label"],
            "true_label": prediction["true_label"],
            "question": question,
            "prompt": prompt,
            "answer": answer,
            "error": error,
            "llm_model": GEMINI_MODEL,
        }
    )

conversation_df = pd.DataFrame(conversation_records)

#### 7.4 Inspect Simulated Conversation

In [25]:
for record in conversation_records:
    print(f"Question {record['turn']}: {record['question']}")
    print(f"Assistant: {record['answer'] or record['error']}")
    print("-" * 80)

Question 1: Why was this researcher assigned to this affiliation?
Assistant: Yu Li was assigned to 'Business Information and Communication Systems' because Yu Li authored publications such as 'Ein Framework zur Modellierung und Analyse von XML-Netzen', 'Simulation von Supply Chains mit Petri-Netzen', and 'Prozessorientierte Komposition von Diensten in der Doktorandenausbildung'. These publications are also associated with 'Business Information and Communication Systems' through the ':publishes' relationship. Yu Li's homepage, identified by the number +49 (721) 608 4548, was also a contributing factor.
--------------------------------------------------------------------------------
Question 2: Which graph relationship was most important for this prediction?
Assistant: The graph relationship most important for this prediction was "Yu Li --:homepage--> +49 (721) 608 4548", with an importance score of 0.739.
--------------------------------------------------------------------------------
Q

## 8. Evaluate Conversational Explanations

This section evaluates whether the generated conversational explanations are grounded in the provided GNNExplainer evidence. The evaluation is manual but structured: each answer is rated according to groundedness, usefulness, uncertainty handling, and hallucination.

#### 8.1 Define Evaluation Criteria

In [26]:
evaluation_criteria = pd.DataFrame(
    [
        {
            "criterion": "Groundedness",
            "question": "Does the answer only use information contained in the evidence table?",
            "score": "1 = yes, 0 = no",
        },
        {
            "criterion": "Usefulness",
            "question": "Does the answer help understand the model prediction?",
            "score": "1 = yes, 0 = no",
        },
        {
            "criterion": "Uncertainty handling",
            "question": "Does the answer clearly state when the evidence is weak or incomplete?",
            "score": "1 = yes, 0 = no",
        },
        {
            "criterion": "Hallucination-free",
            "question": "Does the answer avoid unsupported graph facts or causal claims?",
            "score": "1 = yes, 0 = no",
        },
    ]
)

evaluation_criteria

,criterion,question,score
0,Groundedness,Does the answer only use information contained...,"1 = yes, 0 = no"
1,Usefulness,Does the answer help understand the model pred...,"1 = yes, 0 = no"
2,Uncertainty handling,Does the answer clearly state when the evidenc...,"1 = yes, 0 = no"
3,Hallucination-free,Does the answer avoid unsupported graph facts ...,"1 = yes, 0 = no"


#### 8.2 Create Multi-Rater Evaluation Template

In [27]:
evaluators = [
    "Evaluator_1",
    # "Evaluator_2",
    # "Evaluator_3",
]

score_columns = [
    "grounded",
    "useful",
    "handles_uncertainty",
    "hallucination_free",
]

llm_evaluation = conversation_df[["turn", "question", "answer"]].merge(
    pd.DataFrame({"evaluator": evaluators}),
    how="cross",
)

for column in score_columns:
    llm_evaluation[column] = pd.NA

llm_evaluation["notes"] = ""

llm_evaluation

,turn,question,answer,evaluator,grounded,useful,handles_uncertainty,hallucination_free,notes
0,1,Why was this researcher assigned to this affil...,Yu Li was assigned to 'Business Information an...,Evaluator_1,<NA>,<NA>,<NA>,<NA>,
1,2,Which graph relationship was most important fo...,The graph relationship most important for this...,Evaluator_1,<NA>,<NA>,<NA>,<NA>,
2,3,Is this evidence strong enough to trust the pr...,"Yes, the evidence is strong enough to trust th...",Evaluator_1,<NA>,<NA>,<NA>,<NA>,
3,4,Are there any weak or uninformative facts in t...,"Yes, there are a couple of facts that appear w...",Evaluator_1,<NA>,<NA>,<NA>,<NA>,
4,5,Can you summarize the explanation in one sente...,The prediction is primarily supported by Yu Li...,Evaluator_1,<NA>,<NA>,<NA>,<NA>,
5,6,What can you not conclude from this evidence?,"From this evidence, you cannot conclude:\n\n* ...",Evaluator_1,<NA>,<NA>,<NA>,<NA>,


#### 8.3 Fill Scores Manually

Each evaluator independently fills the score columns:

- `1` = criterion fulfilled
- `0` = criterion not fulfilled

The `notes` column can be used for short justifications or unclear cases.

In [28]:
# Turn 1 evaluation by Evaluator_1
llm_evaluation.loc[
    (llm_evaluation["evaluator"] == "Evaluator_1") & (llm_evaluation["turn"] == 1),
    ["grounded", "useful", "handles_uncertainty", "hallucination_free"],
] = [1, 1, 0, 1]

# Turn 2 evaluation by Evaluator_1
llm_evaluation.loc[
    (llm_evaluation["evaluator"] == "Evaluator_1") & (llm_evaluation["turn"] == 2),
    ["grounded", "useful", "handles_uncertainty", "hallucination_free"],
] = [1, 1, 0, 1]

# Turn 3 evaluation by Evaluator_1
llm_evaluation.loc[
    (llm_evaluation["evaluator"] == "Evaluator_1") & (llm_evaluation["turn"] == 3),
    ["grounded", "useful", "handles_uncertainty", "hallucination_free"],
] = [1, 1, 0, 1]

# Turn 4 evaluation by Evaluator_1
llm_evaluation.loc[
    (llm_evaluation["evaluator"] == "Evaluator_1") & (llm_evaluation["turn"] == 4),
    ["grounded", "useful", "handles_uncertainty", "hallucination_free"],
] = [1, 1, 1, 1]

# Turn 5 evaluation by Evaluator_1
llm_evaluation.loc[
    (llm_evaluation["evaluator"] == "Evaluator_1") & (llm_evaluation["turn"] == 5),
    ["grounded", "useful", "handles_uncertainty", "hallucination_free"],
] = [1, 1, 0, 1]

# Turn 6 evaluation by Evaluator_1
llm_evaluation.loc[
    (llm_evaluation["evaluator"] == "Evaluator_1") & (llm_evaluation["turn"] == 6),
    ["grounded", "useful", "handles_uncertainty", "hallucination_free"],
] = [1, 1, 1, 1]

In [29]:
llm_evaluation

,turn,question,answer,evaluator,grounded,useful,handles_uncertainty,hallucination_free,notes
0,1,Why was this researcher assigned to this affil...,Yu Li was assigned to 'Business Information an...,Evaluator_1,1,1,0,1,
1,2,Which graph relationship was most important fo...,The graph relationship most important for this...,Evaluator_1,1,1,0,1,
2,3,Is this evidence strong enough to trust the pr...,"Yes, the evidence is strong enough to trust th...",Evaluator_1,1,1,0,1,
3,4,Are there any weak or uninformative facts in t...,"Yes, there are a couple of facts that appear w...",Evaluator_1,1,1,1,1,
4,5,Can you summarize the explanation in one sente...,The prediction is primarily supported by Yu Li...,Evaluator_1,1,1,0,1,
5,6,What can you not conclude from this evidence?,"From this evidence, you cannot conclude:\n\n* ...",Evaluator_1,1,1,1,1,


#### 8.4 Summarize Evaluation Scores

In [30]:
llm_evaluation_summary = (
    llm_evaluation[score_columns].astype(float).mean().rename("mean_score").to_frame()
)

(llm_evaluation_summary * 100).round(1)

,mean_score
grounded,100.0
useful,100.0
handles_uncertainty,33.3
hallucination_free,100.0


#### 8.5 Summarize Scores by Evaluator

In [31]:
llm_evaluation_by_evaluator = llm_evaluation.groupby("evaluator")[score_columns].mean()

(llm_evaluation_by_evaluator * 100).round(1)

,grounded,useful,handles_uncertainty,hallucination_free
evaluator,,,,
Evaluator_1,100.0,100.0,33.333333,100.0


## 9. Save Conversational Explanation Outputs

Save the prediction metadata, evidence table, generated prompts, generated answers, and manual LLM evaluation results for later inspection and reporting.

#### 9.1 Define Output Paths

In [32]:
node_id = prediction["node_id"]

conversation_json_path = CONVERSATIONS_DIR / f"conversation_node_{node_id}.json"
conversation_csv_path = CONVERSATIONS_DIR / f"conversation_node_{node_id}.csv"
evidence_csv_path = CONVERSATIONS_DIR / f"evidence_node_{node_id}.csv"
llm_evaluation_csv_path = CONVERSATIONS_DIR / f"llm_evaluation_node_{node_id}.csv"

#### 8.2 Create Output Payload

In [33]:
payload = {
    "prediction": prediction,
    "evidence": evidence_table.to_dict(orient="records"),
    "conversation": conversation_records,
    "llm_evaluation": llm_evaluation.to_dict(orient="records"),
    "llm_evaluation_summary": llm_evaluation_summary.reset_index().to_dict(
        orient="records"
    ),
}

#### 9.3 Save Files

In [34]:
conversation_json_path.write_text(
    json.dumps(
        payload,
        indent=2,
    )
)

conversation_df.to_csv(
    conversation_csv_path,
    index=False,
)

evidence_table.to_csv(
    evidence_csv_path,
    index=False,
)

llm_evaluation.to_csv(
    llm_evaluation_csv_path,
    index=False,
)

#### 8.4 Inspect Saved Outputs

In [35]:
saved_outputs = pd.Series(
    {
        "Conversation JSON": conversation_json_path,
        "Conversation CSV": conversation_csv_path,
        "Evidence CSV": evidence_csv_path,
        "LLM Evaluation CSV": llm_evaluation_csv_path,
    },
    name="saved_path",
).to_frame()

saved_outputs

,saved_path
Conversation JSON,../results/conversations/conversation_node_567...
Conversation CSV,../results/conversations/conversation_node_567...
Evidence CSV,../results/conversations/evidence_node_5672.csv
LLM Evaluation CSV,../results/conversations/llm_evaluation_node_5...


## 10. Aggregate LLM Evaluation Results

This section loads all previously completed LLM evaluation files and computes aggregate scores across all evaluated predictions.

#### 10.1 Load All Evaluation Files

In [36]:
evaluation_files = sorted(CONVERSATIONS_DIR.glob("llm_evaluation_node_*.csv"))

all_llm_evaluations = []

for path in evaluation_files:
    df = pd.read_csv(path)
    df["source_file"] = path.name
    all_llm_evaluations.append(df)

if not all_llm_evaluations:
    raise FileNotFoundError(f"No LLM evaluation files found in {CONVERSATIONS_DIR}.")

all_llm_evaluation_df = pd.concat(
    all_llm_evaluations,
    ignore_index=True,
)

all_llm_evaluation_df.head()

,turn,question,answer,evaluator,grounded,useful,handles_uncertainty,hallucination_free,notes,source_file
0,1,Why was this researcher assigned to this affil...,"Siegfried Handschuh was assigned to the ""Knowl...",Evaluator_1,1,1,0,1,NaN,llm_evaluation_node_5527.csv
1,2,Which graph relationship was most important fo...,The most important graph relationship for this...,Evaluator_1,1,1,0,1,NaN,llm_evaluation_node_5527.csv
2,3,Is this evidence strong enough to trust the pr...,"Yes, the evidence is strong enough to trust th...",Evaluator_1,1,1,0,1,NaN,llm_evaluation_node_5527.csv
3,4,Are there any weak or uninformative facts in t...,"Yes, there are some facts in the explanation t...",Evaluator_1,1,1,1,1,NaN,llm_evaluation_node_5527.csv
4,5,Can you summarize the explanation in one sente...,Siegfried Handschuh is predicted as 'Knowledge...,Evaluator_1,1,1,0,1,NaN,llm_evaluation_node_5527.csv


In [37]:
len(evaluation_files)

6

#### 10.2 Summarize Overall Scores

In [38]:
score_columns = [
    "grounded",
    "useful",
    "handles_uncertainty",
    "hallucination_free",
]

overall_llm_summary = (
    all_llm_evaluation_df[score_columns]
    .astype(float)
    .mean()
    .mul(100)
    .round(1)
    .rename("percentage")
    .to_frame()
)

overall_llm_summary

,percentage
grounded,100.0
useful,100.0
handles_uncertainty,61.1
hallucination_free,100.0


#### 10.3 Summarize by Node

In [39]:
# Extract node ID from source filename if it is not already present
if "node_id" not in all_llm_evaluation_df.columns:
    all_llm_evaluation_df["node_id"] = (
        all_llm_evaluation_df["source_file"]
        .str.extract(r"llm_evaluation_node_(\d+)\.csv")
        .astype(int)
    )

llm_summary_by_node = (
    all_llm_evaluation_df.groupby("node_id")[score_columns]
    .mean()
    .mul(100)
    .round(1)
    .reset_index()
)

llm_summary_by_node

,node_id,grounded,useful,handles_uncertainty,hallucination_free
0,5527,100.0,100.0,33.3,100.0
1,5535,100.0,100.0,83.3,100.0
2,5538,100.0,100.0,83.3,100.0
3,5541,100.0,100.0,66.7,100.0
4,5544,100.0,100.0,66.7,100.0
5,5672,100.0,100.0,33.3,100.0


#### 10.4 Summarize by Evaluator

In [40]:
llm_summary_by_evaluator = (
    all_llm_evaluation_df.groupby("evaluator")[score_columns].mean().mul(100).round(1)
)

llm_summary_by_evaluator

,grounded,useful,handles_uncertainty,hallucination_free
evaluator,,,,
Evaluator_1,100.0,100.0,61.1,100.0


#### 10.5 Summarize by Question

In [41]:
llm_summary_by_question = (
    all_llm_evaluation_df.groupby("question")[score_columns]
    .mean()
    .mul(100)
    .round(1)
    .reset_index()
)

llm_summary_by_question

,question,grounded,useful,handles_uncertainty,hallucination_free
0,Are there any weak or uninformative facts in t...,100.0,100.0,100.0,100.0
1,Can you summarize the explanation in one sente...,100.0,100.0,33.3,100.0
2,Is this evidence strong enough to trust the pr...,100.0,100.0,66.7,100.0
3,What can you not conclude from this evidence?,100.0,100.0,100.0,100.0
4,Which graph relationship was most important fo...,100.0,100.0,0.0,100.0
5,Why was this researcher assigned to this affil...,100.0,100.0,66.7,100.0


#### 10.6 Compute Overall Quality Score

In [42]:
overall_quality_score = (
    all_llm_evaluation_df[score_columns].astype(float).mean().mean() * 100
)

print(f"Overall LLM explanation quality score: {overall_quality_score:.1f}%")

Overall LLM explanation quality score: 90.3%
